In [ ]:
#
# Import Packages Required for this Notebook
#
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql.types import *
from pyspark.sql.functions import expr, countDistinct, col, lit, when
from notebookutils import mssparkutils # type: ignore
import com.microsoft.spark.fabric # type: ignore
import com.microsoft.spark.fabric.Constants # type: ignore
import re
import os
import msal

print("Successfully imported all packages for this notebook.")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 3, Finished, Available, Finished)

Successfully imported all packages for this notebook.


In [13]:
#
# Get required secrets from the key vault
#
vault_uri = "https://kv-fabric-dev-eastus2.vault.azure.net/"

# Retrieve secret from Key Vault using mssparkutils
TENANT_ID = mssparkutils.credentials.getSecret(vault_uri, "TENANT-ID")
CLIENT_ID = mssparkutils.credentials.getSecret(vault_uri, "CLIENT-ID")
CLIENT_SECRET = mssparkutils.credentials.getSecret(vault_uri, "CLIENT-SECRET-KEY")

# Use the secret securely without printing
print("Secrets retrieved successfully (not displayed for security reasons).")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 15, Finished, Available, Finished)

Secrets retrieved successfully (not displayed for security reasons).


In [14]:
#
# Verify that key vault items cannot be viewed in clear text
#
print(f"The value of the tenant ID is {TENANT_ID}")
print(f"The value of the client ID is {CLIENT_ID}")
print(f"The value of the client secret is {CLIENT_SECRET}")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 16, Finished, Available, Finished)

The value of the tenant ID is [REDACTED]
The value of the client ID is [REDACTED]
The value of the client secret is [REDACTED]


In [11]:
#
# Configure run-time parameters for this notebook
#
file_folder = "bronze"
db_schema = "dbo"
application = "warehouse"
workspace_id = "3ac7ce42-ae74-4e7d-8ac3-5ce8358a30df" ## AdventureWorks Dev
lakehouse_id = "50402dac-ce50-4831-af2b-7d65ca8fe7db" ## AdventureWorks_Lakehouse

warehouse_name = "AdventureWorks_Warehouse"

server_name = "jdbc:sqlserver://znbjxinpfs5u3cxz7s7bppllcq-ilhmootuvz6u5cwdltudlcrq34.database.fabric.microsoft.com"
database_name = "{AdventureWorks_Database-12bf21b8-b14c-46c5-a8d3-1401230ed7d7}"

skip_tables = [
    "DimAccount",
    "DimCurrency",
    "DimCustomer",
    "DimDate",
    "DimDepartmentGroup",
    "DimEmployee",
    "DimGeography",
    "DimOrganization",
    "DimProduct",
    "DimProductCategory",
    "DimProductSubcategory",
    "DimPromotion",
    "DimReseller",
    "DimSalesReason",
    "DimSalesTerritory",
    "DimScenario",
    "FactAdditionalInternationalProductDescription",
    "FactCallCenter",
    "FactCurrencyRate",
    "FactFinance",
    "FactInternetSales",
    "FactInternetSalesReason",
    "FactProductInventory",
    "FactResellerSales",
    "FactSalesQuota",
    "FactSurveyResponse",
    "NewFactCurrencyRate",
    # "ProspectiveBuyer"
]

print("Successfully configured all paramaters for this run.")
print(f"For checkpoint restart, the number of tables skipped is {len(skip_tables)}.")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 13, Finished, Available, Finished)

Successfully configured all paramaters for this run.
For checkpoint restart, the number of tables skipped is 27.


In [5]:
#
# Define the OneLake folder paths
#
folder = "/Files/" + file_folder + "/" + application
folder_path = "abfss://" + workspace_id + "@onelake.dfs.fabric.microsoft.com/" + lakehouse_id + folder

output_folder = "/lakehouse/default/Files/resources/"

base_url = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lakehouse_id}/Files/resources"

print(f"Configured to process files from:\n{folder_path} using database schema '{db_schema}' tables.\n")
print(f"Configured to writes files to:\n{output_folder}\n")
print(f"Configured for image base URL:\n{base_url}")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 7, Finished, Available, Finished)

Configured to process files from:
abfss://3ac7ce42-ae74-4e7d-8ac3-5ce8358a30df@onelake.dfs.fabric.microsoft.com/50402dac-ce50-4831-af2b-7d65ca8fe7db/Files/bronze/warehouse using database schema 'dbo' tables.

Configured to writes files to:
/lakehouse/default/Files/resources/

Configured for image base URL:
https://onelake.dfs.fabric.microsoft.com/3ac7ce42-ae74-4e7d-8ac3-5ce8358a30df/50402dac-ce50-4831-af2b-7d65ca8fe7db/Files/resources


In [16]:
#
# Define the JDBC connection string to the GOLD layer database
#
jdbc_url = "".join([
    f"{server_name};",
    f"databaseName={database_name};",
    "Authentication=ActiveDirectoryServicePrincipal;",
    f"Username={CLIENT_ID};",
    f"Password={CLIENT_SECRET};"
])

print(f"Configured to create GOLD layer tables into the database:\n{jdbc_url}.")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 18, Finished, Available, Finished)

Configured to create GOLD layer tables into the database:
jdbc:sqlserver://znbjxinpfs5u3cxz7s7bppllcq-ilhmootuvz6u5cwdltudlcrq34.database.fabric.microsoft.com;databaseName={AdventureWorks_Database-12bf21b8-b14c-46c5-a8d3-1401230ed7d7};Authentication=ActiveDirectoryServicePrincipal;Username=[REDACTED];Password=[REDACTED];.


In [7]:
#
# Define the function to transform binary image columns into files referenced through a URL
#
def transform_binary_to_image_url(df_gold: DataFrame, output_folder_path: StringType, table_name: StringType, field_name: str, primary_key_name: str) -> DataFrame:

    # Create and initialize a URL string column for the GOLD layer
    df_gold = df_gold.withColumn(f"{field_name}Url", lit(None).cast(StringType()))

    # Get the tables' primary key column to used in generating a unique file name
    if primary_key_name == "UniqueID":
        df_image = df_gold.select(field_name)
        df_image = df_image.withColumn(primary_key_name, expr("uuid()")).collect()
    else:
        df_image = df_gold.select(primary_key_name, field_name).collect()

    # Loop through each row and write the binary column as a GIF file
    image_size_kb = 0
    for row in df_image:
        primary_key = row[primary_key_name]
        image_bytes = row[field_name]
        
        if image_bytes and len(image_bytes) > 0:
            image_size_kb += round(len(image_bytes) / 1024)

            # Define full file path and URL
            file_path = os.path.join(output_folder_path, f"{primary_key}-{field_name}.gif")
            file_url = f"{base_url}/{file_path.replace(output_folder_path, '').lstrip(os.sep)}"

            # Save the binary data to OneLake (create directories if they don't exist)
            os.makedirs(os.path.dirname(file_path), exist_ok=True)
            with open(file_path, "wb") as file:
                file.write(bytearray(image_bytes))
            
            # Populate the "Url" column with the file HTTP URL for the row
            df_gold = df_gold.withColumn(
                f"{field_name}Url", 
                when(
                    col(primary_key_name) == primary_key, 
                    lit(file_url)
                ).otherwise(
                    col(f"{field_name}Url")
                )
            )
    
    # Drop the binary column data from the GOLD layer
    df_gold = df_gold.drop(field_name)

    # Display results
    num_files = len(df_image)
    avg_size = round(image_size_kb / num_files)
    print(
        f"Successfully wrote {num_files:,} files for table {table_name} "
        f"and column {field_name} with an average image size of {avg_size:,}Kb into OneLake."
    )

    return df_gold

print("Successfully created function 'transform_binary_to_image_url'.")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 9, Finished, Available, Finished)

Successfully created function 'transform_binary_to_image_url'.


In [8]:
#
# Define the function to transform the SILVER layer dataframe into GOLD
#
def transform_to_gold(df_silver: DataFrame, output_folder_path: StringType, table_name: StringType) -> DataFrame:
    df_gold = df_silver

    # Get the primary key field name; REQUIRED FOR GENERATING UNIQUE IMAGE FILE NAMES
    unique_counts = df_gold.agg(*(countDistinct(col(c)).alias(c) for c in df_gold.columns))
    primary_key_col = [c for c in df_gold.columns if unique_counts.collect()[0][c] == df_gold.count()]
    if len(primary_key_col) > 0:
        primary_key_name = primary_key_col[0]
    else:
        primary_key_name = "UniqueID"

    # Process each binary column that may exists
    for field in df_gold.schema.fields:
        if isinstance(field.dataType, BinaryType):
            df_gold = transform_binary_to_image_url(df_gold, field.name, primary_key_name)
        else:     
            #
            # Data Type Conversion
            df_gold = df_gold.withColumn(field.name, col(field.name).cast(field.dataType))
            
    return df_gold

print("Successfully created function 'transform_to_gold'.")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 10, Finished, Available, Finished)

Successfully created function 'transform_to_gold'.


In [9]:
#
# Create the Spark session
#
app_name = "CreateDatabaseGoldTables"

# Get the current Spark session
spark = SparkSession.builder \
    .appName(app_name) \
    .getOrCreate()

print(f"Spark session {app_name} has been created successfully.")
print(f"Spark session is set with {spark.conf.get('spark.driver.memory')} driver memory.")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 11, Finished, Available, Finished)

Spark session CreateDatabaseGoldTables has been created successfully.
Spark session is set with 56g driver memory.


In [17]:
#
# Transform & Load the SILVER layer tables into the GOLD layer
#
num_tables = 0

# List all files in the Lakehouse
file_list = spark.read.format("binaryFile").load(folder_path).select("path").orderBy("path").collect()

# Iterate through each file in the Lakehouse to get the table name
for file in file_list:
    file_path = file["path"]
    
    if file_path.endswith(".csv"):  # Ensure the file is a CSV
        # Extract the table name from the file name
        table_name = file_path.split("/")[-1].split(".")[0]
        output_folder_path = output_folder + table_name
        if table_name in skip_tables:
            print(f"Skipping table: {table_name}...\n")
            continue
        else:
            print(f"Processing table {table_name}...")

        # Read the corresponding Warehouse SILVER table into a dataframe
        df = spark.read.synapsesql(f"{warehouse_name}.{db_schema}.{table_name}")

        # Transform the dataframe from SILVER to GOLD
        df_gold = transform_to_gold(df, output_folder_path, table_name)

        # Create the GOLD layer table
        # Write DataFrame to Fabric SQL using AAD authentication
        batch_size = 100
        df_gold = df_gold.repartition(batch_size) 
        df_gold.write \
            .mode("overwrite") \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", table_name) \
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .option("batchsize", f"{batch_size}") \
            .save()
        num_rows = df_gold.count()
        num_tables += 1
        print(f"Successfully saved table {table_name} into the GOLD layer with {num_rows:,} rows.\n")

print(f"\n✅ Successfully executed ETL for {num_tables:,} GOLD layer tables into database {database_name}.")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 19, Finished, Available, Finished)

Skipping table: DimAccount...

Skipping table: DimCurrency...

Skipping table: DimCustomer...

Skipping table: DimDate...

Skipping table: DimDepartmentGroup...

Skipping table: DimEmployee...

Skipping table: DimGeography...

Skipping table: DimOrganization...

Skipping table: DimProduct...

Skipping table: DimProductCategory...

Skipping table: DimProductSubcategory...

Skipping table: DimPromotion...

Skipping table: DimReseller...

Skipping table: DimSalesReason...

Skipping table: DimSalesTerritory...

Skipping table: DimScenario...

Skipping table: FactAdditionalInternationalProductDescription...

Skipping table: FactCallCenter...

Skipping table: FactCurrencyRate...

Skipping table: FactFinance...

Skipping table: FactInternetSales...

Skipping table: FactInternetSalesReason...

Skipping table: FactProductInventory...

Skipping table: FactResellerSales...

Skipping table: FactSalesQuota...

Skipping table: FactSurveyResponse...

Skipping table: NewFactCurrencyRate...

Processing

# Restart Capability
##### The above cell received a Java StackOverflow exception. This led to the development of a checkpoint restart capability that skips tables already loaded successfully. The root-cause of the error appears to be due to this message that appeared in another Fabric web page when an attempt was made to refresh the database tables:

> Fetch response error: Unable to complete the action because your organization’s Fabric compute capacity has exceeded its limits. Try again later.



In [18]:
# Stop the Spark session
# NOTE: frees up limited F2 SKU capacity resources
spark.stop()

print("Spark session has been stopped successfully.")

StatementMeta(, c8418965-36d6-4316-9905-901f5e2a7124, 20, Finished, Available, Finished)

Spark session has been stopped successfully.
